In [13]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from joblib import dump
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

# **Task1**

In [14]:
def clean_dataset_task1(file_path, target_cols, output_file):

    df = pd.read_csv(file_path)
    print("Original Shape:", df.shape)

    # ===============================
    # KEEP TARGET COLUMNS
    # ===============================
    target_df = df[target_cols].copy()
    df = df.drop(columns=target_cols)

    # ===============================
    # HANDLE MISSING VALUES
    # ===============================

    # numeric fill → median
    num_cols = df.select_dtypes(include=np.number).columns
    for col in num_cols:
        df[col] = df[col].fillna(df[col].median())

    # categorical fill → mode or 'None'
    cat_cols = df.select_dtypes(include='object').columns
    for col in cat_cols:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(df[col].mode()[0] if df[col].mode().shape[0] > 0 else "None")

    # ===============================
    # ENCODING
    # ===============================

    # One-hot encode 'MSZoning' if present
    if 'MSZoning' in df.columns:
        df = pd.get_dummies(df, columns=['MSZoning'])

    # label encode remaining categorical columns
    label_encoders = {}
    cat_cols = df.select_dtypes(include='object').columns
    for col in cat_cols:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        label_encoders[col] = le

    # ===============================
    # REMOVE UNWANTED COLUMNS
    # ===============================
    df = df.loc[:, ~df.columns.str.contains('Unnamed')]

    # ===============================
    # CONCAT TARGET BACK
    # ===============================
    df = pd.concat([df, target_df], axis=1)

    # ===============================
    # SAVE CLEANED FILE
    # ===============================
    output_file = file_path.replace(".csv", "_clean.csv")
    df.to_csv(output_file, index=False)
    print("Final Shape:", df.shape)
    print(f"Saved cleaned file as: {output_file}")

    return df

In [15]:
clean_dataset_task1("house_price_classification.csv",["SalePrice", "PriceCategory"],"house_price_classification_clean.csv")

Original Shape: (1460, 83)
Final Shape: (1460, 86)
Saved cleaned file as: house_price_classification_clean.csv


,Id,MSSubClass,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,YrSold,SaleType,SaleCondition,MSZoning_C (all),MSZoning_FV,MSZoning_RH,MSZoning_RL,MSZoning_RM,SalePrice,PriceCategory
0,1,60,65.0,8450,1,0,3,3,0,4,...,2008,8,4,False,False,False,True,False,208500,2
1,2,20,80.0,9600,1,0,3,3,0,2,...,2007,8,4,False,False,False,True,False,181500,1
2,3,60,68.0,11250,1,0,0,3,0,4,...,2008,8,4,False,False,False,True,False,223500,2
3,4,70,60.0,9550,1,0,0,3,0,0,...,2006,8,0,False,False,False,True,False,140000,1
4,5,60,84.0,14260,1,0,0,3,0,2,...,2008,8,4,False,False,False,True,False,250000,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,1456,60,62.0,7917,1,0,3,3,0,4,...,2007,8,4,False,False,False,True,False,175000,1
1456,1457,20,85.0,13175,1,0,3,3,0,4,...,2010,8,4,False,False,False,True,False,210000,2
1457,1458,70,66.0,9042,1,0,3,3,0,4,...,2010,8,4,False,False,False,True,False,266500,2
1458,1459,20,68.0,9717,1,0,3,3,0,4,...,2010,8,4,False,False,False,True,False,142125,1


# **Task2**

In [16]:
train_data = pd.read_csv('house_price_classification_clean.csv')

In [17]:
train_data.head(5)

,Id,MSSubClass,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,YrSold,SaleType,SaleCondition,MSZoning_C (all),MSZoning_FV,MSZoning_RH,MSZoning_RL,MSZoning_RM,SalePrice,PriceCategory
0,1,60,65.0,8450,1,0,3,3,0,4,...,2008,8,4,False,False,False,True,False,208500,2
1,2,20,80.0,9600,1,0,3,3,0,2,...,2007,8,4,False,False,False,True,False,181500,1
2,3,60,68.0,11250,1,0,0,3,0,4,...,2008,8,4,False,False,False,True,False,223500,2
3,4,70,60.0,9550,1,0,0,3,0,0,...,2006,8,0,False,False,False,True,False,140000,1
4,5,60,84.0,14260,1,0,0,3,0,2,...,2008,8,4,False,False,False,True,False,250000,2


In [18]:
y = train_data['PriceCategory']
X = train_data.drop(['Id','PriceCategory'], axis=1)

In [19]:
def task2_split_save_data(X, y, test_size, random_state=42):

    X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=test_size,random_state=random_state)
    train_pct = int((1 - test_size) * 100)
    test_pct = int(test_size * 100)

    train_file = f"house_price_classification_clean_train_{train_pct}.csv"
    test_file = f"house_price_classification_clean_test_{test_pct}.csv"

    train_df = pd.concat([X_train, y_train], axis=1)
    test_df = pd.concat([X_test, y_test], axis=1)

    train_df.to_csv(train_file, index=False)
    test_df.to_csv(test_file, index=False)

    print(f"Saved: {train_file}")
    print(f"Saved: {test_file}")

    return X_train, X_test, y_train, y_test

In [20]:
task2_split_save_data(X,y,0.3)
task2_split_save_data(X,y,0.20)
task2_split_save_data(X,y,0.25)

Saved: house_price_classification_clean_train_70.csv
Saved: house_price_classification_clean_test_30.csv
Saved: house_price_classification_clean_train_80.csv
Saved: house_price_classification_clean_test_20.csv
Saved: house_price_classification_clean_train_75.csv
Saved: house_price_classification_clean_test_25.csv


(      MSSubClass  LotFrontage  LotArea  Street  Alley  LotShape  LandContour  \
 1023         120         43.0     3182       1      0         3            3   
 810           20         78.0    10140       1      0         3            3   
 1384          50         60.0     9060       1      0         3            3   
 626           20         69.0    12342       1      0         0            3   
 813           20         75.0     9750       1      0         3            3   
 ...          ...          ...      ...     ...    ...       ...          ...   
 1095          20         78.0     9317       1      0         0            3   
 1130          50         65.0     7804       1      0         3            3   
 1294          20         60.0     8172       1      0         3            3   
 860           50         55.0     7642       1      0         3            3   
 1126         120         53.0     3684       1      0         3            3   
 
       Utilities  LotConfi

# **Task 3**

In [21]:
def task_3_DT(train_file, test_file, criteria, tree_depth, split_percentage):

    train_data = pd.read_csv(train_file)
    test_data = pd.read_csv(test_file)

    X_train = train_data.drop(columns=["PriceCategory"])
    y_train = train_data["PriceCategory"]

    X_test = test_data.drop(columns=["PriceCategory"])
    y_test = test_data["PriceCategory"]

    dt = DecisionTreeClassifier(criterion=criteria,max_depth=tree_depth,random_state=42)

    dt.fit(X_train, y_train)
    y_pred = dt.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    print(f"DT ({criteria}, {tree_depth}) Accuracy: {acc:.4f}")

    filename = f"/content/DT_{criteria}_{tree_depth}_{split_percentage}.joblib"
    dump(dt, filename)

    return acc


In [22]:
def task_3_SVM(train_file, test_file, kernel, split_percentage):

    train_data = pd.read_csv(train_file)
    test_data = pd.read_csv(test_file)

    X_train = train_data.drop(columns=["PriceCategory"])
    y_train = train_data["PriceCategory"]

    X_test = test_data.drop(columns=["PriceCategory"])
    y_test = test_data["PriceCategory"]

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    svm = SVC(kernel=kernel)
    svm.fit(X_train_scaled, y_train)

    y_pred = svm.predict(X_test_scaled)

    acc = accuracy_score(y_test, y_pred)
    print(f"SVM ({kernel}) Accuracy: {acc:.4f}")

    filename = f"/content/SVM_{kernel}_{split_percentage}.joblib"
    dump(svm, filename)

    return acc

In [ ]:
dt_params = [("gini", 4), ("gini", 5), ("entropy", 4), ("entropy", 5)]
svm_kernels = ["linear", "rbf", "poly"]

train_files = {
    70: "house_price_classification_clean_train_70.csv",
    75: "house_price_classification_clean_train_75.csv",
    80: "house_price_classification_clean_train_80.csv"
}

for split_percentage, train_file in train_files.items():

    test_file = f"house_price_classification_clean_test_{100-split_percentage}.csv"

    print(f"\n===== SPLIT {split_percentage}:{100-split_percentage} =====")

    # Decision Tree
    for crit, depth in dt_params:
        task_3_DT(train_file, test_file, crit, depth, split_percentage)

    # SVM
    for kernel in svm_kernels: 
        task_3_SVM(train_file, test_file, kernel, split_percentage)


===== SPLIT 70:30 =====
DT (gini, 4) Accuracy: 1.0000
DT (gini, 5) Accuracy: 1.0000
DT (entropy, 4) Accuracy: 1.0000
DT (entropy, 5) Accuracy: 1.0000
SVM (linear) Accuracy: 0.9018
SVM (rbf) Accuracy: 0.8516
SVM (poly) Accuracy: 0.7352

===== SPLIT 75:25 =====
DT (gini, 4) Accuracy: 1.0000
DT (gini, 5) Accuracy: 1.0000
DT (entropy, 4) Accuracy: 1.0000
DT (entropy, 5) Accuracy: 1.0000
SVM (linear) Accuracy: 0.9123
SVM (rbf) Accuracy: 0.8521
SVM (poly) Accuracy: 0.7452

===== SPLIT 80:20 =====
DT (gini, 4) Accuracy: 1.0000
DT (gini, 5) Accuracy: 1.0000
DT (entropy, 4) Accuracy: 1.0000
DT (entropy, 5) Accuracy: 1.0000
SVM (linear) Accuracy: 0.9144
SVM (rbf) Accuracy: 0.8630
SVM (poly) Accuracy: 0.7466
